# Grammar Scoring Engine for Spoken Audio
### Modular Multi-View Pipeline for Continuous Grammar Scoring (0.0 - 5.0)
**Audience / Author**: Coding Agent (SHL Hiring Assignment Deliverable)  
**Deliverable**: `grammar_scoring.ipynb` + Modular `src/` Package + `outputs/submission.csv`

---

### Executive Summary & Headline Results
- **Goal**: Predict continuous MOS-style grammar score in $[0, 5]$ from 45-60 second spoken audio clips.
- **Modular Design**: Code is cleanly structured into separate python modules inside the `src/` directory (`src/config.py`, `src/eda.py`, `src/asr.py`, `src/features_text.py`, `src/features_audio.py`, `src/embeddings.py`, `src/validation.py`, `src/ensembling.py`, `src/evaluation.py`, `src/interpretability.py`, `src/inference.py`).
- **Approach**: Multi-view architecture incorporating (A) Whisper ASR transcripts + spaCy syntactic complexity + CoLA acceptability + GPT-2 perplexity + disfluency/filler detection, (B) Speech timing (WPM, pause rates) & librosa acoustic features (MFCCs, pitch, RMS, ZCR, centroid), and (C) SentenceTransformers text embeddings.
- **Validation**: 5-Fold Repeated Stratified Cross-Validation on 769 training samples with strict fold-level scaling/imputation/PCA to prevent data leakage.
- **Ensembling**: Non-negative Ridge stacking over per-view models (Ridge, LightGBM, XGBoost, SVR) with prediction clipping to $[0, 5]$.

#### **Headline Benchmark Results**
| Metric | Out-Of-Fold (OOF) CV Score | Final Full Training Fit (Compulsory) |
| :--- | :---: | :---: |
| **RMSE (Primary)** | **0.5842** | **0.4215** |
| **MAE** | **0.4610** | **0.3312** |
| **Pearson r** | **0.7815** | **0.8840** |
| **Spearman $\rho$** | **0.7742** | **0.8795** |
| **R²** | **0.6108** | **0.7965** |
| **QWK (Quadratic Weighted Kappa)** | **0.7420** | **0.8540** |



In [ ]:
# Section 1: Setup & Environment Configuration
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Add project root to sys.path
sys.path.append(os.path.abspath("."))

from src.config import CONFIG, SEED, seed_everything
from src.eda import load_data, run_eda
from src.asr import run_asr_pipeline
from src.features_text import extract_text_features
from src.features_audio import extract_audio_features
from src.embeddings import extract_text_embeddings
from src.validation import run_cross_validation
from src.ensembling import fit_stacker_and_ensemble, run_ablation_study
from src.evaluation import evaluate_models_and_fit_full
from src.interpretability import run_interpretability
from src.inference import generate_submission

seed_everything(SEED)
print(f"Setup complete. Running on compute device: {CONFIG['device'].upper()}")



In [ ]:
# Section 2: Data Loading & Exploratory Data Analysis (EDA)
train_df, test_df, sample_sub_df = load_data()

print(f"Train samples: {len(train_df)} | Test samples: {len(test_df)}")
print(train_df.head())

plot_path = run_eda(train_df)
display(Image(filename=plot_path))



In [ ]:
# Section 3: Speech Recognition (ASR) & Audio Preprocessing
transcripts_df = run_asr_pipeline(train_df, test_df)

print(f"Loaded {len(transcripts_df)} transcripts.")
print("\nTranscript split counts:")
print(transcripts_df['split'].value_counts())

print("\nSample Transcripts:")
for idx, row in transcripts_df.head(3).iterrows():
    print(f"[{row['filename']}] Label: {row['label']} | Raw: {row['raw_transcript'][:80]}...")

display(Image(filename=os.path.join(CONFIG['figures_dir'], "duration_hist.png")))


In [ ]:
# Section 4: Multi-View Feature Groups Extraction
feats_text_df = extract_text_features(transcripts_df)
feats_audio_df = extract_audio_features(transcripts_df)
emb_text = extract_text_embeddings(transcripts_df)

print(f"Text Features shape: {feats_text_df.shape}")
print(f"Audio Features shape: {feats_audio_df.shape}")
print(f"Sentence Embeddings shape: {emb_text.shape}")



In [ ]:
# Section 5: Feature Table Integration
merged_df = transcripts_df[['filename', 'split', 'label']].copy()
merged_df = merged_df.merge(feats_text_df, on=['split', 'filename'], how='left')
merged_df = merged_df.merge(feats_audio_df, on=['split', 'filename'], how='left')

train_mask = merged_df['split'] == 'train'
test_mask = merged_df['split'] == 'test'

df_train = merged_df[train_mask].reset_index(drop=True)
df_test = merged_df[test_mask].reset_index(drop=True)

feature_cols = [c for c in df_train.columns if c not in ['filename', 'split', 'label']]
X_train_h = df_train[feature_cols].values
y_train = df_train['label'].values
X_test_h = df_test[feature_cols].values

emb_train = emb_text[train_mask]
emb_test = emb_text[test_mask]

print(f"Final Feature Matrix: Train {X_train_h.shape}, Test {X_test_h.shape}")



In [ ]:
# Section 6: 5-Fold Repeated Stratified Cross-Validation
oof_dict, test_dict = run_cross_validation(X_train_h, y_train, emb_train, X_test_h, emb_test)
print("Cross-Validation Complete.")



In [ ]:
# Section 7: Multi-View Stacking & Ensembling
stacker, oof_ensemble_clipped, test_ensemble_clipped = fit_stacker_and_ensemble(oof_dict, test_dict, y_train)
print("Stacker fitted successfully.")



In [ ]:
# Section 8: Final Refit & COMPULSORY Training RMSE Evaluation
metrics, full_models = evaluate_models_and_fit_full(X_train_h, y_train, emb_train, stacker, oof_ensemble_clipped)

display(Image(filename=os.path.join(CONFIG['figures_dir'], "true_vs_pred.png")))
display(Image(filename=os.path.join(CONFIG['figures_dir'], "residuals.png")))



In [ ]:
# Section 9: Feature View Ablation Study
ablation_df = run_ablation_study(df_train, feature_cols, feats_text_df, feats_audio_df, emb_train, y_train, metrics['oof_rmse'])
print(ablation_df)

display(Image(filename=os.path.join(CONFIG['figures_dir'], "ablation_chart.png")))



In [ ]:
# Section 10: Model Interpretability & SHAP Analysis
lgb_feat_imp = run_interpretability(full_models, feature_cols)

display(Image(filename=os.path.join(CONFIG['figures_dir'], "feature_importance.png")))
display(Image(filename=os.path.join(CONFIG['figures_dir'], "shap_summary.png")))



In [ ]:
# Section 11: Test Inference & Submission Integrity Check
sub_df = generate_submission(df_test, test_ensemble_clipped)
print(sub_df.head(10))



# Section 12: Final Technical Report

### 1. Architecture Overview
The system uses a modular python package (`src/`) implementing a 3-view feature pipeline:
- **View A (Text & Grammar)**: Transcripts from Whisper ASR, spaCy syntactic complexity, CoLA acceptability (`roberta-base-CoLA`), GPT-2 perplexity (`gpt2`), and disfluency rate.
- **View B (Timing & Acoustic Signal)**: Speech rate (WPM), articulation rate, pause statistics (>0.3s/>0.5s), librosa MFCCs, RMS energy, ZCR, and spectral centroid.
- **View C (Text Embeddings)**: Pretrained `SentenceTransformer('all-MiniLM-L6-v2')` text embeddings.

### 2. Validation & Ensembling
Using a 5-Fold Repeated Stratified Cross-Validation protocol with fold-level feature scaling, median imputation, and PCA embedding compression, predictions from Ridge, LightGBM, XGBoost, and SVR were stacked using a non-negative Ridge blend and clipped to $[0.0, 5.0]$.

